In [3]:
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

with open('dataset/tiny-shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read(
    )

tokenizer = tiktoken.get_encoding("gpt2")
total_characters = len(text)
total_tokens = len(tokenizer.encode(text))
print(f"Total characters: {total_characters}")
print(f"Total tokens: {total_tokens}")

Total characters: 1115394
Total tokens: 338025


In [4]:

GPT_CONFIG = {
    'vocab_size': 50257,
    'context_length': 1024,
    'emb_dim': 768,
    'n_heads': 12,
    'n_layers': 12,
    'drop_rate': 0.1,
    'qkv_bias': False
}


In [5]:
class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk, dtype=torch.long))
            self.target_ids.append(torch.tensor(target_chunk, dtype=torch.long))
            
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [6]:
def create_data_loader(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(txt, tokenizer, max_length, stride)
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return data_loader

In [7]:
train_ratio = 0.9
train_size = int(train_ratio * len(text))
train_data = text[:train_size]
val_data = text[train_size:]

In [8]:
torch.manual_seed(42)

train_loader = create_data_loader(
    train_data, 
    batch_size=2,
    max_length=GPT_CONFIG['context_length'],
    stride=GPT_CONFIG['context_length'],
    shuffle=True,
    drop_last=True,
    num_workers=0
)
val_loader = create_data_loader(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG['context_length'],
    stride=GPT_CONFIG['context_length'],
    shuffle=False,
    drop_last=True,
    num_workers=0
)

In [9]:
print("Train loader:")
for x, y in train_loader:
    print(f"x: {x.shape}, y: {y.shape}")

print("Validation loader:")
for x, y in val_loader:
    print(f"x: {x.shape}, y: {y.shape}")

Train loader:
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2, 1024]), y: torch.Size([2, 1024])
x: torch.Size([2,

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    outputs = model(input_batch)
    loss = torch.nn.functional.cross_entropy(outputs.flatten(0, 1), target_batch.flatten())
    return loss

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float('nan')  # Return NaN if the data_loader is empty
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, num_batches=None):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches)
    model.train()  # Set the model back to training mode after evaluation
    return train_loss, val_loss

In [ ]:
def generate_and_print(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]  # Assuming pos_emb is a nn.Embedding layer
    encoded_context = tokenizer.encode(start_context).to(device)
    with torch.no_grad():
        token_ids = generate_text(model, idx, max_new_tokens=50, context_size=context_size)
    decoded_text = tokenizer.decode(token_ids)
    print(f"Generated text: {decoded_text}")
    model.train()  # Set the model back to training mode after generation

SyntaxError: incomplete input (3647284223.py, line 1)

In [ ]:
def train_model_simple(model, train_loader, val_loader,
                       optimizer, device, num_epochs, eval_freq, eval_iter, start_context,
                       tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, 0
    
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1
            
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Epoch [{epoch+1}/{num_epochs}], Step [{global_step}], Tokens Seen: {tokens_seen}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        

In [ ]:
torch.manual_seed(42)
model = GPT(GPT_CONFIG).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, 
    train_loader, 
    val_loader, 
    optimizer, 
    device, 
    num_epochs=num_epochs, 
    eval_freq=100, 
    eval_iter=10, 
    start_context="ROMEO: ", 
    tokenizer=tokenizer
)